# Top 20 Momentum Stocks Visualization

Shows the top performers by 6-month momentum for the latest rebalancing date.

In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

In [2]:
# Load the prepared momentum dataset
print("Loading momentum dataset...")
df = pd.read_parquet('data/momentum_prepared/momentum_data.parquet')
print(f"Loaded {len(df):,} rows")

Loading momentum dataset...
Loaded 13,495,461 rows


In [3]:
# Load rebalancing dates
rebalance_dates_df = pd.read_csv('data/momentum_prepared/rebalance_dates.csv')
rebalance_dates_df['rebalance_date'] = pd.to_datetime(rebalance_dates_df['rebalance_date'])
rebalance_dates = rebalance_dates_df['rebalance_date'].tolist()

# Get latest rebalancing date
latest_rebalance = rebalance_dates[-1]
print(f"Latest rebalancing date: {latest_rebalance.date()}")

Latest rebalancing date: 2025-11-25


In [4]:
# Filter for latest rebalancing date and eligible stocks
latest_data = df[(df['date'] == latest_rebalance) & (df['is_eligible'])].copy()
print(f"Eligible stocks on this date: {len(latest_data):,}")

# Get top 20 by momentum rank
top_20 = latest_data.nsmallest(20, 'momentum_rank')[[
    'ticker', 'adj_close', 'momentum_6m', 'momentum_rank', 
    'avg_price_6m', 'avg_dollar_volume_1m'
]].copy()

print(f"\nTop 20 Momentum Stocks:")
top_20

Eligible stocks on this date: 5,480

Top 20 Momentum Stocks:


,ticker,adj_close,momentum_6m,momentum_rank,avg_price_6m,avg_dollar_volume_1m
11462484,STRC,96.290,201.843901,1.0,65.423902,1.439659e+08
13141979,WW,26.740,151.800000,2.0,25.784729,1.273990e+07
74009,ABVX,128.230,21.575704,3.0,64.035952,1.530615e+08
13063835,WOLF,19.860,13.391304,4.0,9.303622,4.720492e+07
8314383,NEGG,77.900,11.833608,5.0,47.919206,4.838286e+07
13420920,ZEPP,29.280,10.712000,6.0,28.551508,1.059329e+07
2121280,CDTX,219.650,8.680476,7.0,77.782421,4.101629e+08
11764177,TERN,29.345,8.589869,8.0,8.405675,1.027388e+08
2142951,CELC,99.080,8.436190,9.0,44.868770,7.876904e+07
2566735,COGT,39.720,7.139344,10.0,14.001865,1.410033e+08


In [5]:
# Display top 20 in a formatted table
print("="*70)
print(f"TOP 20 MOMENTUM STOCKS AS OF {latest_rebalance.date()}")
print("="*70)
print(f"{'Rank':<6} {'Ticker':<8} {'Price':<10} {'6M Mom%':<12} {'Avg Price':<12} {'Avg $Vol':<15}")
print("-"*70)

for idx, row in top_20.iterrows():
    rank = int(row['momentum_rank'])
    ticker = row['ticker']
    price = row['adj_close']
    momentum = row['momentum_6m'] * 100
    avg_price = row['avg_price_6m']
    avg_vol = row['avg_dollar_volume_1m']
    
    print(f"{rank:<6} {ticker:<8} ${price:<9.2f} {momentum:>10.2f}%  ${avg_price:<11.2f} ${avg_vol:>13,.0f}")

TOP 20 MOMENTUM STOCKS AS OF 2025-11-25
Rank   Ticker   Price      6M Mom%      Avg Price    Avg $Vol       
----------------------------------------------------------------------
1      STRC     $96.29       20184.39%  $65.42       $  143,965,917
2      WW       $26.74       15180.00%  $25.78       $   12,739,898
3      ABVX     $128.23       2157.57%  $64.04       $  153,061,506
4      WOLF     $19.86        1339.13%  $9.30        $   47,204,920
5      NEGG     $77.90        1183.36%  $47.92       $   48,382,864
6      ZEPP     $29.28        1071.20%  $28.55       $   10,593,295
7      CDTX     $219.65        868.05%  $77.78       $  410,162,924
8      TERN     $29.34         858.99%  $8.41        $  102,738,804
9      CELC     $99.08         843.62%  $44.87       $   78,769,036
10     COGT     $39.72         713.93%  $14.00       $  141,003,316
11     URGN     $27.69         583.70%  $17.34       $   31,098,154
12     KOD      $22.99         556.86%  $10.57       $   15,270,731
13  

In [6]:
# Create charts for top 5 momentum stocks
print("Creating charts for top 5 momentum stocks...")
top_5_tickers = top_20.head(5)['ticker'].tolist()

# Create subplot figure (5 rows, 1 column)
fig = make_subplots(
    rows=5, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.05,
    subplot_titles=[f"{ticker} - 6M Momentum: {top_20[top_20['ticker']==ticker]['momentum_6m'].values[0]:.1%}" 
                    for ticker in top_5_tickers],
    row_heights=[0.2, 0.2, 0.2, 0.2, 0.2]
)

for i, ticker in enumerate(top_5_tickers, 1):
    # Get data for this ticker (last 6 months for context)
    ticker_data = df[df['ticker'] == ticker].copy()
    ticker_data = ticker_data.sort_values('date')
    
    # Get last 6 months
    last_6m = ticker_data[ticker_data['date'] >= (latest_rebalance - pd.Timedelta(days=180))]
    
    # Add candlestick
    fig.add_trace(
        go.Candlestick(
            x=last_6m['date'],
            open=last_6m['adj_open'],
            high=last_6m['adj_high'],
            low=last_6m['adj_low'],
            close=last_6m['adj_close'],
            name=ticker,
            showlegend=False
        ),
        row=i, col=1
    )
    
    # Update y-axis title for this subplot
    fig.update_yaxes(title_text=f"${ticker}", row=i, col=1)

# Update layout
fig.update_layout(
    title=f'Top 5 Momentum Stocks - {latest_rebalance.date()}',
    height=1400,
    showlegend=False,
    xaxis_rangeslider_visible=False,
    hovermode='x unified'
)

# Remove rangesliders from all subplots
for i in range(1, 6):
    fig.update_xaxes(rangeslider_visible=False, row=i, col=1)

fig.show()

# Export to HTML
fig.write_html('top_momentum_charts.html')
print("\n✓ Charts exported to: top_momentum_charts.html")

Creating charts for top 5 momentum stocks...



✓ Charts exported to: top_momentum_charts.html


In [7]:
# Save top 20 list to CSV
output_csv = Path('data/momentum_prepared/top_20_latest.csv')
top_20.to_csv(output_csv, index=False)
print(f"✓ Top 20 list saved to: {output_csv}")

✓ Top 20 list saved to: data/momentum_prepared/top_20_latest.csv


In [8]:
# Show some statistics about the top 20
print("\n" + "="*70)
print("TOP 20 STATISTICS")
print("="*70)
print(f"Average 6-month momentum: {top_20['momentum_6m'].mean():.1%}")
print(f"Median 6-month momentum: {top_20['momentum_6m'].median():.1%}")
print(f"Best performer: {top_20.iloc[0]['ticker']} at {top_20.iloc[0]['momentum_6m']:.1%}")
print(f"20th place: {top_20.iloc[19]['ticker']} at {top_20.iloc[19]['momentum_6m']:.1%}")
print(f"\nAverage price: ${top_20['adj_close'].mean():.2f}")
print(f"Average 6M avg price: ${top_20['avg_price_6m'].mean():.2f}")
print(f"Average dollar volume: ${top_20['avg_dollar_volume_1m'].mean():,.0f}")


TOP 20 STATISTICS
Average 6-month momentum: 2462.4%
Median 6-month momentum: 648.8%
Best performer: STRC at 20184.4%
20th place: SI at 429.9%

Average price: $64.56
Average 6M avg price: $32.41
Average dollar volume: $346,525,672
